# <center> VAI Store - Engenharia de Variáveis </center>

---

Com base nos **_insights_** da Análise Exploratória de Dados, nosso objetivo nesta etapa é **traduzir os padrões que descobrimos** (como sazonalidade e diferença entre filiais) em _**features**_ que o modelo possa usar para prever a demanda dos produtos.

## Conteúdo

1. [Imports](#imports)
2. [Dataset](#dataset)
3. [Engenharia de Variáveis](#engenharia-de-variaveis)

<a id='imports'></a>
## 1. Imports

In [ ]:
import pandas as pd

In [ ]:
# Caminho dos módulos
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src' / 'utils'))

<a id='dataset'></a>
## 2. Datasets

In [ ]:
df_vendas = pd.read_parquet('../data/treated/vendas_tratado.parquet')

# Leitura dos dados
df_vendas.info()
df_vendas.head()

In [ ]:
df_produtos = pd.read_parquet('../data/treated/produtos_tratado.parquet')

# Leitura dos dados
df_produtos.info()
df_produtos.head()

### 2.1. Mesclando datasets

Para analisar a demanda de forma completa, precisamos conectar as vendas com as características dos produtos (categoria e subcategoria).

In [ ]:
from merge_datasets import merge_datasets

df = merge_datasets(df_vendas, df_produtos, 'SKU')

<a id='engenharia-de-variaveis'></a>
## 3. Engenharia de Variáveis

Visto que o foco é a previsão da demanda diária por produto, vamos agregar esses dados para que cada **linha do nosso dataframe** represente a **demanda diária total de um item em uma filial específica**.

### 3.1. Agregação dos dados

In [ ]:
df = df.groupby(['SKU', 'DATA_ATEND', 'FILIAL']).agg(
    QTD_TOTAL=('QTD_VENDA', 'sum'),
    FATUR_TOTAL=('FATUR_VENDA', 'sum'),
    CLIENTES_UNICOS=('CLI_CPF', 'nunique'),
).reset_index()

### 3.2. Extração de dados temporais

Como dito anteriormente, foi revelado fortes padrões temporais e flutuações claras no faturamento e volume de vendas ao longo do ano. Para **permitir que o modelo identifique esses padrões**, vamos decompor a `DATA_ATEND` em _features_ como `DIA`, `MES`, `ANO` e `DIA_SEMANA`.

In [ ]:
df['DIA'] = df['DATA_ATEND'].dt.day
df['MES'] = df['DATA_ATEND'].dt.month
df['ANO'] = df['DATA_ATEND'].dt.year
df['DIA_SEMANA'] = df['DATA_ATEND'].dt.dayofweek

### 3.3. Criação de flags e índices sazonais

A análise provou que os picos de faturamento em março, novembro e dezembro são ocasionados por eventos festivos (Páscoa e Natal). Além disso, foi mostrado que o 'Bacalhau' tem suas vendas concentradas nessas datas. Por isso, vamos criar variáveis que possibilitam **capturar corretamente a demanda dos produtos em datas festivas**. 

In [ ]:
def vespera_to_bool(data):
    if data['DIA'] == 24 and data['MES'] == 12:
        return 1
    else:
        return 0

df['VESPERA_NATAL'] = df.apply(vespera_to_bool, axis=1)

In [ ]:
def dias_ate_natal(data):
    ano = data.year
    data_natal = pd.to_datetime(f'{ano}-12-25')

    if data > data_natal:
        data_natal = pd.to_datetime(f'{ano + 1}-12-25')
    
    diff = data_natal - data
    return diff.days

df['DIAS_ATE_NATAL'] = df['DATA_ATEND'].apply(dias_ate_natal)

### 3.4. Encoding

A EDA demonstrou que `FILIAL` é um dos **preditores mais importantes na diferença de faturamento**. Portanto, vamos converter a coluna em um formato numérico para que o modelo possa entender e aprender os padrões de venda distintos de cada filial.

In [ ]:
df = pd.concat([
    df.drop('FILIAL', axis=1), 
    pd.get_dummies(df['FILIAL'], drop_first=True).astype(int)], 
    axis=1
)